标准的 sigmoid 函数：
$$
Sigmoid(x) = \frac{1}{1+e^{-x}}
$$

sigmoid函数增加缩放参数 w ：
$$
Sigmoid(wx) = \frac{1}{1+e^{-wx}}
$$

sigmoid函数增加平移参数 b ：
$$
Sigmoid(wx+b) = \frac{1}{1+e^{-(wx+b)}}
$$

逻辑回归的损失函数：Cross Entropy Loss 交叉熵损失
$$
H(p,q) = -\sum_x p(x) \log{q(x)},
$$
其中，$p(x)$是真实分布，$q(x)$是模型预测分布，$H(p,q)$是交叉熵，表示用基于真实分布p的期望来计算基于模型或预测分布q的编码长度

二分类：
$$
Loss = \frac{1}{N} \sum_i -[y_i * \log(p_i) + (1-y_i)*\log(1-p_i)]
$$
其中，$y_i$表示标签，正类为1，负类为0

多分类：
$$
Loss = - \frac{1}{N} \sum_i \sum_{c=1}^M y_{ic} \log(p_{ic})
$$
其中，$M$表示类别的数量；$y_{ic}$表示符号函数0或1，如果样本的真实类别等于c则取1，否则取0；$p_{ic}$表示观测样本属于类别c的预测概率

**梯度下降**：
- **batch gradient descent** 批量梯度下降：用所有训练数据求得每个参数的平均梯度；
- **stochastic gradient descent 随机梯度下降**：每次只从训练集中抽一个样本进行计算更新，单个样本随机性强可以帮助模型跳出局部最小，但训练波动大不稳定；
- **mini-batch gradient descent 小批量梯度下降**：每次从训练集中抽小部分样本进行计算更新。取出的小部分样本叫batch，每一次训练叫step，把训练集所有样本都训练一遍叫epoch

泰坦尼克生存预测

In [1]:
import pandas as pd
import torch

In [2]:
train = pd.read_csv("D:/agent/torch/data/titanic/train.csv")
test = pd.read_csv("D:/agent/torch/data/titanic/test.csv")

print(train.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


one-hot encoding 独热编码

In [3]:
# 独热编码把离散类别变量转换为向量表示
# 对于有 C 个类别的变量，用一个长度为 C 的向量表示，其中只有对应类别的位置是 1，其余为 0

train = train.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
train = train.dropna(subset=["Age"])

# 对 Sex 和 Embarked 做独热编码 (虚拟变量)
train = pd.get_dummies(train, columns=["Sex", "Embarked"], dtype=int)

print(train.head())

   Survived  Pclass   Age  SibSp  Parch     Fare  Sex_female  Sex_male  \
0         0       3  22.0      1      0   7.2500           0         1   
1         1       1  38.0      1      0  71.2833           1         0   
2         1       3  26.0      0      0   7.9250           1         0   
3         1       1  35.0      1      0  53.1000           1         0   
4         0       3  35.0      0      0   8.0500           0         1   

   Embarked_C  Embarked_Q  Embarked_S  
0           0           0           1  
1           1           0           0  
2           0           0           1  
3           0           0           1  
4           0           0           1  


Pytorch中的Dataset类：读取数据和预处理

In [7]:
# 任何自定义数据集都需要继承 torch.utils.data.Dataset，
# 并实现 __len__ 和 __getitem__(idx)两个方法，
# 其中，__len__ 需要返回整个数据集样本的个数
# __getitem__(idx) 需要根据样本的index返回具体的样本
from torch.utils.data import Dataset

class TitanicDataset(Dataset):
    def __init__(self, file_path):
        self.file_path = file_path
        self.mean = {
            "Pclass": 2.236695,
            "Age": 29.699118,
            "SibSp": 0.512605,
            "Parch": 0.431373,
            "Fare": 34.694514,
            "Sex_female": 0.365546,
            "Sex_male": 0.634454,
            "Embarked_C": 0.182073,
            "Embarked_Q": 0.039216,
            "Embarked_S": 0.775910
        }

        self.std = {
            "Pclass": 0.838250,
            "Age": 14.526497,
            "SibSp": 0.929783,
            "Parch": 0.853289,
            "Fare": 52.918930,
            "Sex_female": 0.481921,
            "Sex_male": 0.481921,
            "Embarked_C": 0.386175,
            "Embarked_Q": 0.194244,
            "Embarked_S": 0.417274
        }

        self.data = self._load_data()
        self.feature_size = len(self.data.columns) - 1

    def _load_data(self):
        df = pd.read_csv(self.file_path)
        df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
        df = df.dropna(subset=["Age"])
        df = pd.get_dummies(df, columns=["Sex", "Embarked"], dtype=int)

        # 标准化
        base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
        for i in range(len(base_features)):
            df[base_features[i]] = (df[base_features[i]] - self.mean[base_features[i]]) / self.std[base_features[i]]
        return df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = self.data.drop(columns=["Survived"]).iloc[idx].values
        label = self.data["Survived"].iloc[idx]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

Pytorch中的DataLoader类：将数据分成小批量

In [8]:
# 一般不用自定义 DataLoader 类
# batchsize 设置每次读取数据的大小
# shuffle 设置是否对数据集进行打乱
# num_workers 设置指定多进程加载
from torch.utils.data import DataLoader

dataset = TitanicDataset("D:/agent/torch/data/titanic/train.csv")
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
for inputs, labels in dataloader:
    print(inputs.shape, labels.shape)
    break

torch.Size([32, 10]) torch.Size([32])


nn.Module

In [15]:
# 模型都需要继承自nn.Module，必须实现 __init__ 和 forward 两个方法
# __init__：第一步调用父类的__init__，然后定义模型的参数和其他继承自nn.Module的模块
# forward：定义数据在模块内如何进行传递计算

import torch
from torch import nn

class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)   # nn.Linear也继承自nn.Module，输入为input_dim，输出为一个值

    def forward(self, x):
        return torch.sigmoid(self.linear(x))    # Logistic Regression 输出概率

Optimizer 优化器
```
# 定义优化器，传入模型的参数，并且设置固定的学习率
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(epochs):
    for features,labels in DataLoader(myDataset, batch_size=256):
        optimizer.zero_grad()   ##优化器清理管理参数的梯度值。
        loss = forward_and_compute_loss(features, labels)   ##前向传播并计算loss
        loss.backward()     ##loss反向传播,每个参数计算梯度
        optimizer.step()    ##优化器进行参数更新```

Titanic逻辑回归完整代码

In [21]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import pandas as pd

class TitanicDataset(Dataset):
    def __init__(self, file_path):
        self.file_path = file_path
        self.mean = {
            "Pclass": 2.236695,
            "Age": 29.699118,
            "SibSp": 0.512605,
            "Parch": 0.431373,
            "Fare": 34.694514,
            "Sex_female": 0.365546,
            "Sex_male": 0.634454,
            "Embarked_C": 0.182073,
            "Embarked_Q": 0.039216,
            "Embarked_S": 0.775910
        }

        self.std = {
            "Pclass": 0.838250,
            "Age": 14.526497,
            "SibSp": 0.929783,
            "Parch": 0.853289,
            "Fare": 52.918930,
            "Sex_female": 0.481921,
            "Sex_male": 0.481921,
            "Embarked_C": 0.386175,
            "Embarked_Q": 0.194244,
            "Embarked_S": 0.417274
        }

        self.data = self._load_data()
        self.feature_size = len(self.data.columns) - 1

    def _load_data(self):
        df = pd.read_csv(self.file_path)
        df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
        df = df.dropna(subset=["Age"])
        df = pd.get_dummies(df, columns=["Sex", "Embarked"], dtype=int)

        # 标准化
        base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
        for i in range(len(base_features)):
            df[base_features[i]] = (df[base_features[i]] - self.mean[base_features[i]]) / self.std[base_features[i]]
        return df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = self.data.drop(columns=["Survived"]).iloc[idx].values
        label = self.data["Survived"].iloc[idx]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)


class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)   # nn.Linear也继承自nn.Module，输入为input_dim，输出为一个值

    def forward(self, x):
        return torch.sigmoid(self.linear(x))    # Logistic Regression 输出概率

# 读取数据
train_dataset = TitanicDataset("D:/agent/torch/data/titanic/train.csv")
validation_dataset = TitanicDataset("D:/agent/torch/data/titanic/validation.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LogisticRegressionModel(input_dim=train_dataset.feature_size)
model.to(device)
model.train()   # 切换到训练模式，dropout随机丢弃神经元，用batch统计量进行标准化

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100

for epoch in range(epochs):
    correct = 0
    step = 0
    total_loss = 0
    for features, labels in DataLoader(train_dataset, batch_size=256, shuffle=True):
        step += 1
        features = features.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()   # 清空梯度
        outputs = model(features).squeeze()   # 去掉维度=1的维度，匹配loss计算的格式
        correct += torch.sum((outputs >= 0.5) == labels)    # 根据概率统计预测正确的数量
        loss = torch.nn.functional.binary_cross_entropy(outputs, labels)
        total_loss += loss.item()
        loss.backward()     # 计算梯度
        optimizer.step()    # 更新参数
    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/step:.4f}")

model.eval()    # 切换到推理阶段，关闭dropout，使用全局均值和方差
with torch.no_grad():   # 关闭自动求导，在这个模块内所有张量运算都不会构建计算图，不会记录梯度
    correct = 0
    for features, labels in DataLoader(validation_dataset, batch_size=256, shuffle=False):
        features = features.to(device)
        labels = labels.to(device)
        outputs = model(features).squeeze()
        correct += torch.sum((outputs >= 0.5) == labels)
    print(f"Validation Accuracy: {correct / len(validation_dataset)}")

Epoch 1, Loss: 0.6906
Epoch 11, Loss: 0.5333
Epoch 21, Loss: 0.4940
Epoch 31, Loss: 0.4753
Epoch 41, Loss: 0.4659
Epoch 51, Loss: 0.4618
Epoch 61, Loss: 0.4528
Epoch 71, Loss: 0.4507
Epoch 81, Loss: 0.4503
Epoch 91, Loss: 0.4478
Validation Accuracy: 0.8311688303947449
